In [1]:
pip install pandas pyodbc SQLAlchemy openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 3.4 MB/s  0:00:003.1 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.9/616.9 kB 5.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [SQLAlchemy]━━━━━━━ 2/3 [SQLAlchemy]
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pyodbc

pyodbc.drivers()

['ODBC Driver 18 for SQL Server']

In [11]:
import pyodbc

# Database connection details
server = "localhost,1433"
database = "TrainingDB"
username = "sa"
password = "StrongPassw0rd!2026"

# Create connection using pyodbc
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "TrustServerCertificate=yes;"
)

print("Connected successfully")

Connected successfully


In [13]:
cursor = conn.cursor()

# Execute SQL query
cursor.execute("SELECT TOP 5 * FROM Sales.Customers")

# Fetch rows from the result
rows = cursor.fetchall()

# Print each row
for row in rows:
    print(row)

(1, 'John Smith', 'USA', 85)
(2, 'Mary Johnson', 'USA', 90)
(3, 'Robert Brown', 'USA', 78)
(4, 'Linda Davis', 'USA', 92)
(5, 'James Wilson', 'USA', 88)


In [15]:
import pandas as pd

query = """
SELECT 
    CustomerName,
    Score
FROM Sales.Customers
"""

# Read SQL query result into a pandas DataFrame
df = pd.read_sql(query, conn)

# View first rows
print(df.head())

   CustomerName  Score
0    John Smith     85
1  Mary Johnson     90
2  Robert Brown     78
3   Linda Davis     92
4  James Wilson     88


/tmp/ipykernel_187485/3754829514.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [16]:
print(df.head())
print(df.info())
print(df.isnull().sum())

   CustomerName  Score
0    John Smith     85
1  Mary Johnson     90
2  Robert Brown     78
3   Linda Davis     92
4  James Wilson     88
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   CustomerName  10 non-null     str  
 1   Score         10 non-null     int64
dtypes: int64(1), str(1)
memory usage: 292.0 bytes
None
CustomerName    0
Score           0
dtype: int64


In [17]:
# Remove spaces from column names and make them lowercase
df.columns = df.columns.str.strip().str.lower()

# Convert year column to integer
df["year"] = pd.to_numeric(df["year"], errors="coerce")

# Convert value column to numeric
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Remove rows where value is missing
df = df.dropna(subset=["value"])

# Remove duplicate records
df = df.drop_duplicates()

# Create a new column for value category
df["value_category"] = df["value"].apply(
    lambda x: "High" if x > 20 else "Normal"
)

print(df.head())

KeyError: 'year'

In [23]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

server = "localhost,1433"
database = "TrainingDB"
username = "sa"
password = "StrongPassw0rd!2026"

# Build SQL Server connection string
connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "TrustServerCertificate=yes;"
)

# Encode the connection string safely
encoded_conn_str = quote_plus(connection_string)

# Create SQLAlchemy engine
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={encoded_conn_str}")

print("SQLAlchemy engine created")

SQLAlchemy engine created


In [24]:
query = """
SELECT 
    CustomerName,
    Score
FROM Sales.Customers
"""

df = pd.read_sql(query, engine)

print(df.head())

   CustomerName  Score
0    John Smith     85
1  Mary Johnson     90
2  Robert Brown     78
3   Linda Davis     92
4  James Wilson     88


In [25]:
df.to_sql(
    name="cleaned_weo_data",   # New table name in SQL Server
    con=engine,                # SQLAlchemy connection engine
    if_exists="replace",       # Replace table if it already exists
    index=False                # Do not save pandas index as a column
)

print("Cleaned data saved to SQL Server")

Cleaned data saved to SQL Server
